# ManūAI job runner

Launches `scripts/agent_job.sh` in the background with this kernel's Python (the `ManuAI` env), under `caffeinate -i` so the Mac doesn't idle-sleep. The job keeps running if this notebook or VS Code is closed.

- Log: `data/logs/agent_job.log`  
- Stop it: `kill $(cat data/logs/agent_job.pid)` in a terminal, or run the **Stop** cell below.
- Refuses to start a second job while one is running (16 GB machine: one heavy job at a time).

The job currently in `scripts/agent_job.sh` is the **overnight run**: the full feature cache (~15 min, ~34 GB), then the first full AST training run (~8–9 h). Run it when you won't need the laptop.

In [4]:
import os, sys, time, signal, subprocess, pathlib
repo = pathlib.Path.cwd()
while not (repo / 'manuai').is_dir() and repo != repo.parent:
    repo = repo.parent
logs = repo / 'data' / 'logs'; logs.mkdir(parents=True, exist_ok=True)
log, pidf = logs / 'agent_job.log', logs / 'agent_job.pid'

def running() -> int | None:
    try:
        pid = int(pidf.read_text())
    except Exception:
        return None
    try:  # reap our own finished child so it doesn't linger as a zombie
        if os.waitpid(pid, os.WNOHANG)[0] == pid:
            return None
    except ChildProcessError:
        pass
    try:
        os.kill(pid, 0); return pid
    except OSError:
        return None

if (pid := running()):
    print(f'⏳ A job is already running (PID {pid}) — not starting another.')
else:
    with open(log, 'a') as f:
        f.write(f'\n===== launch {time.ctime()} python={sys.executable}\n')
    env = dict(os.environ, PYTHON=sys.executable, PYTHONUNBUFFERED='1')
    p = subprocess.Popen(['/usr/bin/caffeinate', '-i', '/bin/bash', str(repo / 'scripts' / 'agent_job.sh')],
                         cwd=repo, env=env, stdout=open(log, 'a'), stderr=subprocess.STDOUT,
                         start_new_session=True)
    pidf.write_text(str(p.pid))
    print(f'🚀 Started PID {p.pid}; logging to {log}')


🚀 Started PID 61483; logging to /Users/harrywills/Desktop/University/AIML339/ManuAI.nosync/data/logs/agent_job.log


In [5]:
# Status: last lines of the log
print('running PID:', running())
print(''.join(open(log).readlines()[-25:]) if log.exists() else 'no log yet')


running PID: 61483

 98%|█████████▊| 52/53 [00:14<00:00,  3.70it/s]

100%|██████████| 53/53 [00:14<00:00,  3.27it/s]
                                               

                                               
{'eval_loss': 3.0196661949157715, 'eval_accuracy': 0.08333333333333333, 'eval_balanced_accuracy': 0.08333333333333333, 'eval_macro_f1': 0.048941051813406376, 'eval_weighted_f1': 0.04894105181340639, 'eval_macro_auc': 0.6064538690476191, 'eval_n': 840, 'eval_runtime': 46.5847, 'eval_samples_per_second': 18.032, 'eval_steps_per_second': 1.138, 'epoch': 1.0, 'device_mem_gb': 1.65}

100%|██████████| 40/40 [05:57<00:00,  4.28s/it]

100%|██████████| 53/53 [00:34<00:00,  3.27it/s]

                                               
                                               
{'train_runtime': 358.0315, 'train_samples_per_second': 3.575, 'train_steps_per_second': 0.112, 'train_loss': 3.028136110305786, 'epoch': 1.0, 'device_mem_gb': 1.65}

100%|██████████| 40/40 [05:58<00:00,  8.95s

### Stop the running job (not run by *Run All* — the cell is guarded)

In [6]:
STOP = False  # set True and run this cell to stop the job
if STOP and (pid := running()):
    os.killpg(os.getpgid(pid), signal.SIGTERM); print('stopped', pid)
